In [10]:
import json
import time
from urllib.request import urlopen

INPUT_SWITCHES_IP_ADDRESS = '169.254.10.10'          # set to 1,2,1,1 by default
OUTPUT_SWITCHES_IP_ADDRESS = '169.254.12.12'         # set to 2,1,1,1 for vna and 2,1,2,1 for spa
PROGRAMMABLE_ATTENUATOR_IP_ADDRESS = '169.254.11.11' # set to 25 dB by default

def send_url_command(ip_address, cmd_to_send):
    url = f"http://{ip_address}/:{cmd_to_send}"
    for attempt in range(3):
        try:
            with urlopen(url, timeout=3) as response:
                pte_return = response.read()
            if len(pte_return) > 100:
                print(f"Error, command not found: {url}")
                return "Invalid Command!"
            return pte_return.decode('utf-8').strip()
        except Exception as network_err:
            if attempt == 2:
                print(f"Hardware Network Warning: No response from {ip_address}. Error: {network_err}")
                return "No Response!"
            time.sleep(0.05)

def set_switch_state(ip_address, A, B, C, D):
    state_byte = 128 + int(A-1) + (2 * int(B-1)) + (4 * int(C-1)) + (8 * int(D-1))
    return send_url_command(ip_address, f"SETP={state_byte}")

def set_programmable_attenuator(attenuation):
    send_url_command(PROGRAMMABLE_ATTENUATOR_IP_ADDRESS, f"SETATT={attenuation}")

def get_switch_state(ip_address):
    raw_reply = send_url_command(ip_address, "SWPORT?")
    if any(err in raw_reply for err in ["Error", "No Response"]):
        return {"status": "Disconnected or Error", "raw_reply": raw_reply}
    try:
        state_byte = int(raw_reply.strip())
        working_value = state_byte - 128
        return {
            "status": "Online",
            "raw_byte": state_byte,
            "A": working_value % 2 + 1,
            "B": (working_value // 2) % 2 + 1,
            "C": (working_value // 4) % 2 + 1,
            "D": (working_value // 8) % 16 + 1
        }
    except ValueError:
        return {"status": f"Malformed hardware response: {raw_reply}"}

def get_attenuator_state(ip_address):
    # The direct URL that successfully delivers the information
    url = f"http://{ip_address}/ATT?"
    
    try:
        with urlopen(url, timeout=2.0) as response:
            # Read bytes, decode to string, and remove whitespace/newlines
            attenuation_value = response.read().decode('utf-8').strip()
            return float(attenuation_value)
            
    except URLError as e:
        print(f"Connection failed: {e.reason}")
        return None
    except ValueError:
        print(f"Parsing Error: Could not convert response to number: {attenuation_value}")
        return None

In [15]:
get_attenuator_state(PROGRAMMABLE_ATTENUATOR_IP_ADDRESS) # get current value of programmable attenuator

25.0

In [19]:
set_programmable_attenuator(25) # set programmable attenuator to 25 dB

In [12]:
get_switch_state(INPUT_SWITCHES_IP_ADDRESS)
# should be {'status': 'Online', 'raw_byte': 130, 'A': 1, 'B': 2, 'C': 1, 'D': 1}

{'status': 'Online', 'raw_byte': 130, 'A': 1, 'B': 2, 'C': 1, 'D': 1}

In [9]:
get_switch_state(OUTPUT_SWITCHES_IP_ADDRESS)
# should be {'status': 'Online', 'raw_byte': 129, 'A': 2, 'B': 1, 'C': 1, 'D': 1} for vna
# or {'status': 'Online', 'raw_byte': 133, 'A': 2, 'B': 1, 'C': 2, 'D': 1} for spa

{'status': 'Online', 'raw_byte': 133, 'A': 2, 'B': 1, 'C': 2, 'D': 1}

In [11]:
# set input switch state to standard mode
set_switch_state(INPUT_SWITCHES_IP_ADDRESS,1,2,1,1)

'1'

In [21]:
# set output state for vna mode
set_switch_state(OUTPUT_SWITCHES_IP_ADDRESS,2,1,1,1)

'1'

In [20]:
# set output state for spa mode
set_switch_state(OUTPUT_SWITCHES_IP_ADDRESS,2,1,2,1)

'1'